In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from tkinter import ttk
from PIL import Image, ImageTk
import numpy as np
import joblib
import os

class PlantClassifierGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Plant 이미지 분류기 (Apple vs Spear)")
        self.root.geometry("800x600")
        
        # 모델 로드
        try:
            self.model = joblib.load('plant_logistic_model.pkl')
            self.scaler = joblib.load('plant_scaler.pkl')
            print("모델 로드 성공!")
        except:
            # 간단한 모델 사용 (파일이 없을 경우)
            try:
                self.model = joblib.load('plant_model.pkl')
                self.scaler = joblib.load('plant_scaler.pkl')
                print("대체 모델 로드 성공!")
            except:
                messagebox.showerror("오류", "모델 파일을 찾을 수 없습니다.\n먼저 모델을 학습시켜주세요.")
                root.destroy()
                return
        
        self.image_path = None
        self.setup_ui()
    
    def setup_ui(self):
        # 스타일 설정
        style = ttk.Style()
        style.theme_use('clam')
        
        # 메인 프레임
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # 제목
        title_label = ttk.Label(main_frame, text="Plant 이미지 분류기", 
                               font=('Arial', 20, 'bold'))
        title_label.grid(row=0, column=0, columnspan=2, pady=10)
        
        # 이미지 표시 프레임
        self.image_frame = ttk.LabelFrame(main_frame, text="이미지", padding="10")
        self.image_frame.grid(row=1, column=0, padx=10, pady=10)
        
        # 이미지 레이블
        self.image_label = ttk.Label(self.image_frame, text="이미지를 선택하세요")
        self.image_label.grid(row=0, column=0)
        
        # 결과 표시 프레임
        result_frame = ttk.LabelFrame(main_frame, text="분류 결과", padding="10")
        result_frame.grid(row=1, column=1, padx=10, pady=10, sticky=(tk.N, tk.S))
        
        # 결과 레이블들
        self.result_label = ttk.Label(result_frame, text="예측 결과: -", 
                                     font=('Arial', 14, 'bold'))
        self.result_label.grid(row=0, column=0, pady=5)
        
        # 확률 표시
        self.prob_frame = ttk.Frame(result_frame)
        self.prob_frame.grid(row=1, column=0, pady=10)
        
        ttk.Label(self.prob_frame, text="Apple 확률:", font=('Arial', 12)).grid(row=0, column=0, sticky=tk.W)
        self.apple_prob = ttk.Label(self.prob_frame, text="0%", font=('Arial', 12))
        self.apple_prob.grid(row=0, column=1, padx=10)
        
        ttk.Label(self.prob_frame, text="Spear 확률:", font=('Arial', 12)).grid(row=1, column=0, sticky=tk.W)
        self.spear_prob = ttk.Label(self.prob_frame, text="0%", font=('Arial', 12))
        self.spear_prob.grid(row=1, column=1, padx=10)
        
        # 진행률 표시
        self.progress_apple = ttk.Progressbar(result_frame, length=200, mode='determinate')
        self.progress_apple.grid(row=2, column=0, pady=5)
        
        self.progress_spear = ttk.Progressbar(result_frame, length=200, mode='determinate')
        self.progress_spear.grid(row=3, column=0, pady=5)
        
        # 버튼 프레임
        button_frame = ttk.Frame(main_frame)
        button_frame.grid(row=2, column=0, columnspan=2, pady=20)
        
        # 버튼들
        ttk.Button(button_frame, text="이미지 선택", 
                  command=self.select_image).grid(row=0, column=0, padx=5)
        ttk.Button(button_frame, text="분류하기", 
                  command=self.classify_image).grid(row=0, column=1, padx=5)
        ttk.Button(button_frame, text="초기화", 
                  command=self.reset).grid(row=0, column=2, padx=5)
        
        # 상태바
        self.status_bar = ttk.Label(self.root, text="준비됨", relief=tk.SUNKEN)
        self.status_bar.grid(row=1, column=0, sticky=(tk.W, tk.E))
        
        # 그리드 가중치 설정
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main_frame.columnconfigure(0, weight=1)
        main_frame.columnconfigure(1, weight=1)
    
    def select_image(self):
        """이미지 파일 선택"""
        filename = filedialog.askopenfilename(
            title="이미지 선택",
            filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp"), 
                      ("All files", "*.*")]
        )
        
        if filename:
            self.image_path = filename
            self.display_image(filename)
            self.status_bar.config(text=f"이미지 로드됨: {os.path.basename(filename)}")
    
    def display_image(self, image_path):
        """이미지 표시"""
        try:
            # 이미지 로드 및 크기 조정
            img = Image.open(image_path)
            img = img.resize((300, 300), Image.Resampling.LANCZOS)
            
            # tkinter에서 사용할 수 있는 형식으로 변환
            photo = ImageTk.PhotoImage(img)
            
            # 이미지 표시
            self.image_label.config(image=photo)
            self.image_label.image = photo  # 참조 유지
            
        except Exception as e:
            messagebox.showerror("오류", f"이미지 로드 실패: {str(e)}")
    
    def extract_features(self, image_path):
        """이미지에서 특징 추출"""
        try:
            # 이미지 로드
            img = Image.open(image_path)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            img_array = np.array(img)
            
            # 특징 추출
            feature_vector = []
            
            # RGB 채널별 통계
            for channel in range(3):
                channel_data = img_array[:, :, channel]
                feature_vector.extend([
                    channel_data.mean(),
                    channel_data.std(),
                    channel_data.min(),
                    channel_data.max(),
                    np.median(channel_data),
                ])
            
            # HSV Hue 히스토그램
            hsv_img = img.convert('HSV')
            hsv_array = np.array(hsv_img)
            hue_hist, _ = np.histogram(hsv_array[:, :, 0], bins=12, range=(0, 255))
            feature_vector.extend(hue_hist.tolist())
            
            return np.array(feature_vector).reshape(1, -1)
            
        except Exception as e:
            print(f"특징 추출 오류: {e}")
            return None
    
    def classify_image(self):
        """이미지 분류"""
        if not self.image_path:
            messagebox.showwarning("경고", "먼저 이미지를 선택하세요.")
            return
        
        self.status_bar.config(text="분류 중...")
        self.root.update()
        
        try:
            # 특징 추출
            features = self.extract_features(self.image_path)
            
            if features is None:
                messagebox.showerror("오류", "특징 추출 실패")
                return
            
            # 특징 수가 맞는지 확인
            expected_features = self.scaler.n_features_in_
            if features.shape[1] != expected_features:
                # 간단한 특징만 사용 (RGB 평균, 표준편차)
                img = Image.open(self.image_path)
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                img_array = np.array(img)
                
                features = np.array([
                    img_array[:,:,0].mean(), img_array[:,:,0].std(),
                    img_array[:,:,1].mean(), img_array[:,:,1].std(),
                    img_array[:,:,2].mean(), img_array[:,:,2].std(),
                ]).reshape(1, -1)
            
            # 정규화
            features_scaled = self.scaler.transform(features)
            
            # 예측
            prediction = self.model.predict(features_scaled)[0]
            probabilities = self.model.predict_proba(features_scaled)[0]
            
            # 클래스 인덱스 확인
            classes = self.model.classes_
            apple_idx = np.where(classes == 'apple')[0][0] if 'apple' in classes else 0
            spear_idx = np.where(classes == 'spear')[0][0] if 'spear' in classes else 1
            
            apple_prob = probabilities[apple_idx]
            spear_prob = probabilities[spear_idx]
            
            # 결과 표시
            self.result_label.config(text=f"예측 결과: {prediction.upper()}")
            
            # 예측된 클래스에 따라 색상 변경
            if prediction == 'apple':
                self.result_label.config(foreground='red')
            else:
                self.result_label.config(foreground='blue')
            
            # 확률 표시
            self.apple_prob.config(text=f"{apple_prob:.1%}")
            self.spear_prob.config(text=f"{spear_prob:.1%}")
            
            # 진행률 바 업데이트
            self.progress_apple['value'] = apple_prob * 100
            self.progress_spear['value'] = spear_prob * 100
            
            self.status_bar.config(text="분류 완료!")
            
        except Exception as e:
            messagebox.showerror("오류", f"분류 실패: {str(e)}")
            self.status_bar.config(text="오류 발생")
    
    def reset(self):
        """초기화"""
        self.image_path = None
        self.image_label.config(image='', text="이미지를 선택하세요")
        self.result_label.config(text="예측 결과: -", foreground='black')
        self.apple_prob.config(text="0%")
        self.spear_prob.config(text="0%")
        self.progress_apple['value'] = 0
        self.progress_spear['value'] = 0
        self.status_bar.config(text="준비됨")

# 메인 실행
if __name__ == "__main__":
    root = tk.Tk()
    app = PlantClassifierGUI(root)
    root.mainloop()

C:\Users\User\anaconda3\envs\TF210Py310\lib\site-packages\scipy\__init__.py:169: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


모델 로드 성공!
